In [1]:
## Libraries
import numpy as np
import pandas as pd
import librosa
from pathlib import Path
from tqdm import tqdm
from scipy.fftpack import dct

In [9]:
## Sampling rate
sr=16000
## No of frequency bins
n_fft=512
hop_length=160
win_length=320
## No of bark features we want to compress our model into
n_bark=22

In [10]:
def bark_filterbank(sr, n_fft, n_bark=22):
    freqs = np.linspace(0, sr/2, n_fft//2 + 1)
    bark = 6 * np.arcsinh(freqs / 600)
    bark_bins = np.linspace(bark.min(), bark.max(), n_bark + 1)  # n_bark+1 not +2
    
    fb = np.zeros((n_bark, len(freqs)))
    for i in range(n_bark):
        lower = bark_bins[i]
        center = bark_bins[i + 1]
        if i + 2 < len(bark_bins):
            upper = bark_bins[i + 2]
        else:
            upper = bark.max()
        
        for j, b in enumerate(bark):
            if lower <= b <= center:
                fb[i, j] = (b - lower) / (center - lower + 1e-8)
            elif center < b <= upper:
                fb[i, j] = (upper - b) / (upper - center + 1e-8)
    
    # Normalize so weights sum to 1
    fb = fb / (fb.sum(axis=1, keepdims=True) + 1e-8)
    return fb

bark_fb = bark_filterbank(sr, n_fft)

In [11]:
def extract_rnnoise_features(audio):
    # STFT
    stft = librosa.stft(audio, n_fft=n_fft, hop_length=hop_length, 
                         win_length=win_length, window='hann')
    magnitude = np.abs(stft)
    
    # === FIX: Use squared magnitude for energy ===
    bark_energy = np.dot(bark_fb, magnitude**2)
    log_bark = np.log1p(bark_energy)
    
    # BFCC (DCT of log Bark energies)
    bfcc = dct(log_bark, type=2, axis=0, norm='ortho')
    
    # === FIX: Don't transpose yet ===
    # Shape: (22, n_frames)
    
    # Delta features of FIRST 6 BFCCs only
    delta = librosa.feature.delta(bfcc[:6], order=1)
    delta_delta = librosa.feature.delta(bfcc[:6], order=2)
    
    '''# === PITCH FEATURES (simplified approximation) ===
    # Compute pitch using autocorrelation
    f0, voiced_flag, voiced_probs = librosa.pyin(
        audio, fmin=100, fmax=400, sr=sr,
        frame_length=win_length, hop_length=hop_length
    )
    pitch_period = np.nan_to_num(f0, nan=0.0)
    pitch_period = 1.0 / (pitch_period + 1e-8)  # Convert F0 to period
    pitch_period = pitch_period.reshape(1, -1)  # (1, n_frames)
    
    # Simplified pitch correlation approximation using voiced probability
    pitch_corr = voiced_probs.reshape(1, -1)  # Use as proxy
    pitch_corr = np.tile(pitch_corr, (6, 1))  # Repeat for 6 bands'''

    # === PITCH FEATURES (Speed Optimized for Deadline) ===
    f0 = librosa.yin(
        audio, fmin=100, fmax=400, sr=sr,
        frame_length=win_length, hop_length=hop_length
    )
    pitch_period = np.nan_to_num(f0, nan=0.0)
    pitch_period = 1.0 / (pitch_period + 1e-8)  
    pitch_period = pitch_period.reshape(1, -1)  
    
    # Mock voiced probabilities to preserve 42-feature tensor shape
    voiced_probs = np.where(f0 > 0, 1.0, 0.0) 
    pitch_corr = voiced_probs.reshape(1, -1)  
    pitch_corr = np.tile(pitch_corr, (6, 1))
    
    '''# === NON-STATIONARITY (spectral flux approximation) ===
    spectral_flux = np.sqrt(np.sum(np.diff(magnitude**2, axis=1)**2, axis=0))
    spectral_flux = spectral_flux.reshape(1, -1)'''

    # === NON-STATIONARITY (spectral flux approximation) ===
    # np.diff reduces time dimension by 1. 
    flux_diff = np.diff(magnitude**2, axis=1)
    spectral_flux = np.sqrt(np.sum(flux_diff**2, axis=0))
    
    # Pad with one zero at the start to restore original n_frames length
    spectral_flux = np.pad(spectral_flux, (1, 0), mode='constant')
    spectral_flux = spectral_flux.reshape(1, -1)
    
    # Concatenate all features
    # bfcc: (22, n_frames), delta: (6, n_frames), etc.
    features = np.concatenate([
        bfcc,              # 22
        delta,             # 6
        delta_delta,       # 6
        pitch_corr[:6],    # 6 (placeholder)
        pitch_period,      # 1
        spectral_flux      # 1
    ], axis=0)
    
    return features.T  # Now transpose to (n_frames, n_features)

In [12]:
def compute_rnnoise_gain(clean_audio, noisy_audio, bark_fb=bark_fb):
    min_len = min(len(clean_audio), len(noisy_audio))
    clean_audio = clean_audio[:min_len]
    noisy_audio = noisy_audio[:min_len]
    
    # STFT - MUST use SAME parameters as feature extraction!
    clean_stft = librosa.stft(clean_audio, n_fft=n_fft, hop_length=hop_length,
                               win_length=win_length, window='hann')
    noisy_stft = librosa.stft(noisy_audio, n_fft=n_fft, hop_length=hop_length,
                              win_length=win_length, window='hann')
    
    # === FIX: Use squared magnitude for energy ===
    clean_mag = np.abs(clean_stft)**2
    noisy_mag = np.abs(noisy_stft)**2
    
    # Bark band energy
    clean_bark = np.dot(bark_fb, clean_mag)
    noisy_bark = np.dot(bark_fb, noisy_mag)
    
    # === FIX: Take SQUARE ROOT for correct IRM ===
    gain = np.sqrt(clean_bark / (noisy_bark + 1e-8))
    
    # Clip to [0, 1]
    gain = np.clip(gain, 0, 1)
    
    return gain.T  # (n_bark, n_frames) -> (n_frames, n_bark)


In [13]:
SEQ_LEN = 50
STRIDE = 25
CONTEXT = 2
BATCH_SIZE = 16

In [29]:
clean_folder='aligned_clean_dataset'
noisy_folder='noisy_dataset'

clean_files=list(Path(clean_folder).rglob('*.wav'))
noisy_files=list(Path(noisy_folder).rglob('*.wav'))

print(f"Found {len(clean_files)} clean files")
print(f"Found {len(noisy_files)} noisy files")

Found 2783 clean files
Found 2783 noisy files


In [14]:
# ============================================================
# STACK FRAMES (add temporal context)
# ============================================================
def stack_frames(X, context=CONTEXT):
    """Add temporal context by stacking neighboring frames."""
    num_frames, num_features = X.shape
    
    # Pad with edge replication
    padded = np.pad(X, ((context, context), (0, 0)), mode='edge')
    
    # Stack context frames for each position
    stacked = np.zeros((num_frames, num_features * (2 * context + 1)))
    
    for i in range(num_frames):
        stacked[i] = padded[i : i + 2*context + 1].reshape(-1)
    
    return stacked

In [31]:
# ============================================================
# CREATE OVERLAPPING SEQUENCES
# ============================================================
def create_overlapping_sequences(features, gain_targets, seq_len=SEQ_LEN, stride=STRIDE):
    """Create overlapping sequences for RNN training."""
    X, Y = [], []
    total_frames = len(features)
    
    for start in range(0, total_frames - seq_len, stride):
        end = start + seq_len
        X.append(features[start:end])
        Y.append(gain_targets[start:end])
    
    return np.array(X, dtype=np.float32), np.array(Y, dtype=np.float32)

In [15]:
# ============================================================
# MEAN/STD COMPUTATION (run once, then load)
# ============================================================
MEAN_PATH = 'mean.npy'
STD_PATH = 'std.npy'

def compute_normalization_stats(noisy_files, extract_fn, stack_fn, context=CONTEXT):
    """Compute mean and std over all training features (run once)."""
    all_features = []
    
    print("Computing normalization statistics...")
    for noisy_path in tqdm(noisy_files):
        noisy_audio, _ = librosa.load(noisy_path, sr=sr)
        features = extract_fn(noisy_audio)
        features = stack_fn(features, context=context)
        all_features.append(features)
    
    all_features = np.vstack(all_features)
    mean = np.mean(all_features, axis=0)
    std = np.std(all_features, axis=0) + 1e-8  # Avoid division by zero
    
    return mean, std

In [16]:
def load_or_compute_stats():
    """Load existing stats or compute if not exist."""
    if Path(MEAN_PATH).exists() and Path(STD_PATH).exists():
        print("Loading pre-computed statistics...")
        mean = np.load(MEAN_PATH)
        std = np.load(STD_PATH)
    else:
        print("Statistics file not found. Computing...")
        mean, std = compute_normalization_stats(noisy_files, extract_rnnoise_features, stack_frames)
        np.save(MEAN_PATH, mean)
        np.save(STD_PATH, std)
        print(f"Saved statistics to {MEAN_PATH}, {STD_PATH}")
    
    return mean, std

# Load stats
mean, std = load_or_compute_stats()
print(f"Feature dimension after stacking: {len(mean)}")

Loading pre-computed statistics...
Feature dimension after stacking: 210


In [34]:
# ============================================================
# DATA GENERATOR (Corrected)
# ============================================================
def data_generator(clean_files, noisy_files, mean, std, 
                   extract_fn, gain_fn, batch_size=BATCH_SIZE,
                   seq_len=SEQ_LEN, stride=STRIDE, context=CONTEXT):
    """
    Keras-compatible data generator for RNNoise training.
    """
    # Build clean file lookup dictionary ONCE outside the loop
    clean_dict = {file.stem: file for file in clean_files}
    
    while True:
        X_batch, Y_batch = [], []
        
        # Removed tqdm to prevent VS Code terminal buffer crashes during model.fit
        for noisy_path in noisy_files:
            noisy_name = noisy_path.stem
            
            # === CRITICAL FIX: Exact Match ===
            # Because aligned_clean_dataset uses identical filenames to noisy_dataset
            clean_key = noisy_name
            
            if clean_key not in clean_dict:
                continue  # Silently skip to prevent IDE crashes
            
            clean_path = clean_dict[clean_key]
            
            # Load audio files
            try:
                clean_audio, _ = librosa.load(clean_path, sr=sr)
                noisy_audio, _ = librosa.load(noisy_path, sr=sr)
            except Exception:
                continue
            
            # Ensure same length
            min_len = min(len(clean_audio), len(noisy_audio))
            clean_audio = clean_audio[:min_len]
            noisy_audio = noisy_audio[:min_len]
            
            # Extract features and targets
            features = extract_fn(noisy_audio)
            gain_targets = gain_fn(clean_audio, noisy_audio)
            
            # Align frames
            min_frames = min(len(features), len(gain_targets))
            features = features[:min_frames]
            gain_targets = gain_targets[:min_frames]
            
            # Add temporal context
            features = stack_frames(features, context=context)
            
            # NORMALIZE FEATURES (not targets - they are already [0,1])
            features = (features - mean) / std
            
            # Ensure correct dtype
            features = features.astype(np.float32)
            gain_targets = gain_targets.astype(np.float32)
            
            # Create overlapping sequences
            X_seq, Y_seq = create_overlapping_sequences(
                features, gain_targets, seq_len=seq_len, stride=stride
            )
            
            # Yield batches
            for x, y in zip(X_seq, Y_seq):
                X_batch.append(x)
                Y_batch.append(y)
                
                if len(X_batch) == batch_size:
                    # Reshape for RNN: (batch, seq_len, features)
                    yield np.array(X_batch), np.array(Y_batch)
                    X_batch, Y_batch = [], []

In [35]:
# ============================================================
# TEST CELL - Run this AFTER all functions are defined
# ============================================================

# Step 1: Create generator
train_gen = data_generator(
    clean_files=clean_files,
    noisy_files=noisy_files,
    mean=mean,
    std=std,
    extract_fn=extract_rnnoise_features,
    gain_fn=compute_rnnoise_gain
)

# Step 2: Get one batch
X_batch, Y_batch = next(train_gen)
print(f"X shape: {X_batch.shape}")  # Expected: (16, 50, 210)
print(f"Y shape: {Y_batch.shape}")  # Expected: (16, 50, 22)
print(f"X min: {X_batch.min():.4f}, max: {X_batch.max():.4f}")
print(f"Y min: {Y_batch.min():.4f}, max: {Y_batch.max():.4f}")

X shape: (16, 50, 210)
Y shape: (16, 50, 22)
X min: -5.2682, max: 9.1854
Y min: 0.0008, max: 1.0000


/tmp/ipykernel_1142/3677332356.py:36: UserWarning: With fmin=100.000, sr=16000 and frame_length=320, less than two periods of fmin fit into the frame, which can cause inaccurate pitch detection. Consider increasing to fmin=100.000 or frame_length=321.
  f0 = librosa.yin(


In [2]:
## GRU
import tensorflow as tf
from tensorflow.keras.models import Model
from tensorflow.keras.layers import Input, GRU, Dense, TimeDistributed,Concatenate,BatchNormalization, Activation, Conv1D, Add
import soundfile as sf

I0000 00:00:1774236947.216788   22138 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
I0000 00:00:1774236947.218655   22138 cudart_stub.cc:31] Could not find cuda drivers on your machine, GPU will not be used.
I0000 00:00:1774236947.492124   22138 cpu_feature_guard.cc:227] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 AVX512F AVX512_VNNI FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
I0000 00:00:1774236950.037031   22138 port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.

In [37]:
def build_gru_model(seq_len=50, feature_dim=210, output_dim=22):
    inputs=Input(shape=(seq_len,feature_dim))

    x=Conv1D(64, 3,padding='same')(inputs)
    x=BatchNormalization()(x)
    x=Activation('relu')(x)

    skip = x # Store the processed, normalized state

    x=Conv1D(64, 3,padding='same')(x)
    x=BatchNormalization()(x) # ADDED missing BN
    x=Add()([x,skip]) # Clean ResNet-style skip
    x=Activation('relu')(x)

    x=GRU(128, return_sequences=True)(x)
    x=GRU(64,return_sequences=True)(x)
    x=Dense(64,activation='relu')(x)

    outputs=Dense(output_dim,activation='sigmoid')(x)
    return Model(inputs, outputs)

In [38]:
## Build model
model=build_gru_model()
model.summary()

E0000 00:00:1774232038.415657    1142 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 50, 210)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 50, 64)    │     40,384 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 50, 64)    │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 50, 64)    │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 50, 64)    │     12,352 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 50, 64)    │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 50, 64)    │          0 │ batch_normalizat… │
│                     │                   │            │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 50, 64)    │          0 │ add[0][0]         │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru (GRU)           │ (None, 50, 128)   │     74,496 │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ (None, 50, 64)    │     37,248 │ gru[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 50, 64)    │      4,160 │ gru_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 50, 22)    │      1,430 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 170,582 (666.34 KB)

 Trainable params: 170,326 (665.34 KB)

 Non-trainable params: 256 (1.00 KB)

In [39]:
model.compile(optimizer=tf.keras.optimizers.Adam(learning_rate=1e-5),loss='mse',metrics=['mae','accuracy'])

In [62]:
import tensorflow as tf

# 1. Increase learning rate (Keras 3 compatible method)
model.optimizer.learning_rate = 1e-4

# 2. Setup Callbacks (Metrics + Weights)
csv_logger = tf.keras.callbacks.CSVLogger('training_history.csv', append=True)
checkpoint = tf.keras.callbacks.ModelCheckpoint(
    filepath='rnnoise_checkpoint.keras',
    save_weights_only=False,
    save_best_only=False,  
    save_freq='epoch'      
)

print("Resuming training. Metrics and weights continuously saved to disk...")

# 3. Execute training with both callbacks
model.fit(
    data_generator(
        clean_files, 
        noisy_files, 
        mean, 
        std, 
        extract_fn=extract_rnnoise_features,  
        gain_fn=compute_rnnoise_gain,       
        batch_size=32, 
        seq_len=50
    ),
    steps_per_epoch=len(noisy_files) // 32, 
    epochs=15,  
    callbacks=[csv_logger, checkpoint]
)

# 4. Final explicit save
model.save('rnnoise_blower_suppressor_final.keras')
print("Training complete.")

Resuming training. Metrics and weights continuously saved to disk...
Epoch 1/15


/tmp/ipykernel_1142/3677332356.py:36: UserWarning: With fmin=100.000, sr=16000 and frame_length=320, less than two periods of fmin fit into the frame, which can cause inaccurate pitch detection. Consider increasing to fmin=100.000 or frame_length=321.
  f0 = librosa.yin(


86/86 ━━━━━━━━━━━━━━━━━━━━ 12s 139ms/step - accuracy: 0.1699 - loss: 0.0539 - mae: 0.1732
Epoch 2/15
86/86 ━━━━━━━━━━━━━━━━━━━━ 11s 128ms/step - accuracy: 0.1722 - loss: 0.0523 - mae: 0.1704
Epoch 3/15
86/86 ━━━━━━━━━━━━━━━━━━━━ 10s 120ms/step - accuracy: 0.1763 - loss: 0.0515 - mae: 0.1688
Epoch 4/15
86/86 ━━━━━━━━━━━━━━━━━━━━ 11s 132ms/step - accuracy: 0.1744 - loss: 0.0512 - mae: 0.1680
Epoch 5/15
86/86 ━━━━━━━━━━━━━━━━━━━━ 10s 120ms/step - accuracy: 0.1727 - loss: 0.0515 - mae: 0.1687
Epoch 6/15
86/86 ━━━━━━━━━━━━━━━━━━━━ 10s 122ms/step - accuracy: 0.1808 - loss: 0.0496 - mae: 0.1645
Epoch 7/15
86/86 ━━━━━━━━━━━━━━━━━━━━ 10s 113ms/step - accuracy: 0.1758 - loss: 0.0518 - mae: 0.1676
Epoch 8/15
86/86 ━━━━━━━━━━━━━━━━━━━━ 10s 119ms/step - accuracy: 0.1758 - loss: 0.0494 - mae: 0.1649
Epoch 9/15
86/86 ━━━━━━━━━━━━━━━━━━━━ 9s 109ms/step - accuracy: 0.1730 - loss: 0.0523 - mae: 0.1692
Epoch 10/15
86/86 ━━━━━━━━━━━━━━━━━━━━ 10s 121ms/step - accuracy: 0.1806 - loss: 0.0498 - mae: 0.1650
E

In [19]:
def predict_with_chunks(features, model, seq_len=50):
    outputs=[]
    for i in range(0, len(features)-seq_len+1,seq_len):
        chunk=features[i:i+seq_len]
        chunk=np.expand_dims(chunk, axis=0)
        pred=model.predict(chunk, verbose=0)[0]
        outputs.append(pred)
    return np.vstack(outputs)

In [20]:
def pad_features(features, seq_len=50):
    T=len(features)
    remainder=T% seq_len

    if remainder!=0:
        pad_len=seq_len-remainder
        pad=np.zeros((pad_len,features.shape[1]))

        features=np.vstack([features,pad])
    return features,T

In [21]:
## Prediction
def enhance_audio(noisy_audio, model, bark_fb, mean=mean, std=std, n_fft=512, hop_length=hop_length, win_length=win_length):
    ## STFT
    noisy_stft=librosa.stft(noisy_audio, n_fft=n_fft, hop_length=hop_length, window='hann')

    mag=np.abs(noisy_stft)
    phase=np.angle(noisy_stft)


    ## Feature extraction
    features=extract_rnnoise_features(noisy_audio)
    features=stack_frames(features, context=2)

    ## Normalize
    features=(features-mean)/std
    features,original_len=pad_features(features,50)
    pred_gain=predict_with_chunks(features, model, seq_len=50)
    pred_gain = pred_gain[:original_len]
    ## Convert bark->fft
    fft_gain=np.dot(bark_fb.T, pred_gain.T)

    ## Match shape
    fft_gain=fft_gain[:, :mag.shape[1]]

    ## Apply gain
    enhanced_mag=mag*fft_gain

    ## Reconstruct complex stft
    enhanced_stft=enhanced_mag*np.exp(1j*phase)

    ## Inverse stft
    enhanced_audio=librosa.istft(enhanced_stft,hop_length=hop_length,win_length=win_length,window='hann')

    return enhanced_audio

In [23]:
import tensorflow as tf

# Load the final trained model from disk
model_path = 'rnnoise_blower_suppressor_final.keras'

try:
    model = tf.keras.models.load_model(model_path)
    print(f"Successfully loaded model from {model_path}")
    model.summary() # Optional: Verify architecture loaded correctly
except ValueError as e:
    print(f"Error loading model: {e}")
except FileNotFoundError:
    print(f"Error: {model_path} not found. Ensure the file exists in your current directory.")

E0000 00:00:1774237334.292419   22138 cuda_platform.cc:52] failed call to cuInit: INTERNAL: CUDA error: Failed call to cuInit: UNKNOWN ERROR (303)


Successfully loaded model from rnnoise_blower_suppressor_final.keras


Model: "functional"

┏━━━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┳━━━━━━━━━━━━┳━━━━━━━━━━━━━━━━━━━┓
┃ Layer (type)        ┃ Output Shape      ┃    Param # ┃ Connected to      ┃
┡━━━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━╇━━━━━━━━━━━━╇━━━━━━━━━━━━━━━━━━━┩
│ input_layer         │ (None, 50, 210)   │          0 │ -                 │
│ (InputLayer)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d (Conv1D)     │ (None, 50, 64)    │     40,384 │ input_layer[0][0] │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalization │ (None, 50, 64)    │        256 │ conv1d[0][0]      │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation          │ (None, 50, 64)    │          0 │ batch_normalizat… │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ conv1d_1 (Conv1D)   │ (None, 50, 64)    │     12,352 │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ batch_normalizatio… │ (None, 50, 64)    │        256 │ conv1d_1[0][0]    │
│ (BatchNormalizatio… │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ add (Add)           │ (None, 50, 64)    │          0 │ batch_normalizat… │
│                     │                   │            │ activation[0][0]  │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ activation_1        │ (None, 50, 64)    │          0 │ add[0][0]         │
│ (Activation)        │                   │            │                   │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru (GRU)           │ (None, 50, 128)   │     74,496 │ activation_1[0][… │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ gru_1 (GRU)         │ (None, 50, 64)    │     37,248 │ gru[0][0]         │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense (Dense)       │ (None, 50, 64)    │      4,160 │ gru_1[0][0]       │
├─────────────────────┼───────────────────┼────────────┼───────────────────┤
│ dense_1 (Dense)     │ (None, 50, 22)    │      1,430 │ dense[0][0]       │
└─────────────────────┴───────────────────┴────────────┴───────────────────┘

 Total params: 511,236 (1.95 MB)

 Trainable params: 170,326 (665.34 KB)

 Non-trainable params: 256 (1.00 KB)

 Optimizer params: 340,654 (1.30 MB)

In [24]:
import os
import shutil
from pathlib import Path
import soundfile as sf
import librosa

# 1. Create the 'comparing' folder
output_dir = Path('comparing')
output_dir.mkdir(exist_ok=True)

# Define directories
noisy_dir = Path('noisy_dataset')
clean_dir = Path('aligned_clean_dataset')

# 2. Get the first two existing files dynamically
existing_files = list(noisy_dir.glob('*.wav'))[7:9]

if len(existing_files) < 2:
    print(f"Error: You need at least 2 files in {noisy_dir}.")
else:
    for audio_path in existing_files:
        filename = audio_path.name
        print(f"Processing: {filename}")
        
        # Load noisy audio
        noisy_audio, _ = librosa.load(audio_path, sr=sr)
        
        # Save original noisy version
        sf.write(output_dir / f"noisy_{filename}", noisy_audio, sr)
        
        # Fetch and copy the aligned clean target
        clean_path = clean_dir / filename
        if clean_path.exists():
            shutil.copyfile(clean_path, output_dir / f"clean_{filename}")
        else:
            print(f"Warning: Ground truth {filename} not found in {clean_dir}.")
        
        # Enhance using your trained GRU model
        enhanced_audio = enhance_audio(
            noisy_audio=noisy_audio, 
            model=model, 
            bark_fb=bark_fb, 
            mean=mean, 
            std=std
        )
        
        # Save enhanced version
        sf.write(output_dir / f"enhanced_{filename}", enhanced_audio, sr)

    print(f"\nSUCCESS! Check the '{output_dir}' folder for noisy, clean, and enhanced comparisons.")

Processing: FK64_05_snr10_noisefreesound_community-industrial-blower-recording-31979.wav


/tmp/ipykernel_22138/3677332356.py:36: UserWarning: With fmin=100.000, sr=16000 and frame_length=320, less than two periods of fmin fit into the frame, which can cause inaccurate pitch detection. Consider increasing to fmin=100.000 or frame_length=321.
  f0 = librosa.yin(


Processing: FE29_06_snr15_noisefreesound_community-computer-fan-75947.wav

SUCCESS! Check the 'comparing' folder for noisy, clean, and enhanced comparisons.
